# Miniproyecto 3 — ¿Cuánto vale el preentrenamiento?

## BETO sobre reseñas turísticas en español (Rest-Mex 2025)

Maestría · Universidad Icesi · Curso de Procesamiento de Lenguaje Natural

Autores: Juan José Aguado · Juan David Cruz · Juan Diego Ramírez

---

> **Sobre la reutilización del corpus.** Este notebook trabaja sobre el mismo corpus de los
> Miniproyectos 1 y 2 y reproduce su análisis exploratorio **íntegro y sin modificar**, para
> que sea autosuficiente. **Las Secciones 1 a 4 (entorno, corpus, EDA y protocolo) son las del
> Miniproyecto 1, sin modificar**: misma submuestra, mismo split, misma función de evaluación, de
> modo que los números de las tres entregas son directamente comparables. Lo nuevo empieza en
> §4.4: pasamos de entrenar desde cero a **reutilizar un modelo preentrenado en español**, BETO,
> siguiendo el notebook de la Sesión 3.

## 0. El problema

Dada una reseña turística escrita en español sobre un destino mexicano, predecir la polaridad
(1 a 5 estrellas) que le asignó su autor, usando únicamente el texto.

En las dos entregas anteriores, entrenando todo desde cero, ningún modelo superó con claridad a
**TF-IDF + Regresión Logística (macro-F1 0,524)**: ni la LSTM, ni la BiLSTM con vectores de spaCy,
ni el Transformer implementado a mano. El Miniproyecto 1 dejó fuera a BETO porque el
*fine-tuning* se vería en una entrega posterior. Es esta, y la pregunta es:

> **¿Cuánto vale el preentrenamiento sobre esta tarea, y qué parte del modelo hay que ajustar
> para cobrarlo?**

Comparamos las tres formas de usar BERT que muestra el notebook guía —como extractor congelado
con una cabeza lineal, como extractor congelado con una cabeza MLP, y con *fine-tuning* completo—
sobre el mismo split y las mismas métricas que las entregas anteriores.

> **H1.** BETO con *fine-tuning* supera a TF-IDF, y la ganancia se concentra en **2★ y 3★**, las
> fronteras finas donde todos los modelos anteriores fallan.
>
> **H2.** Congelado, BETO **no** supera a TF-IDF: su representación no está organizada por
> sentimiento. Lo que importa es descongelar, no la complejidad de la cabeza.
>
> **H3.** Con ~2.000 reseñas, BETO iguala al Transformer desde cero de MP2 entrenado con 32.000.

### Mapa del notebook

| Sección | Contenido |
|---|---|
| 1–4.3 | Entorno, corpus, **EDA** y protocolo (submuestra, splits, métricas, baselines) · *heredados del Miniproyecto 1, sin modificar* |
| 4.4–4.7 | Lo propio de BERT: dependencias y configuración, **tokenizador WordPiece**, `MAX_LEN`, conjuntos |
| 5 | **Técnica A** — BETO congelado + cabeza lineal |
| 6 | **Técnica B** — BETO congelado + cabeza MLP propia |
| 7 | **Técnica C** — *fine-tuning* completo |
| 8 | Comparación de las tres técnicas y de las tres entregas |
| 9 | ¿Cuántas capas hay que descongelar? LR discriminativa |
| 10 | ¿Qué modelo preentrenado? BETO vs. mBERT vs. DistilBETO |
| 11 | Curva de eficiencia de datos |
| 12 | LoRA: *fine-tuning* eficiente en parámetros |
| 13 | Tarea de control: `Type` |
| 14 | ¿Qué aprendió el *fine-tuning*? Embeddings e *Integrated Gradients* |
| 15 | Demo con pruebas de estrés |
| 16 | Análisis de errores |
| 17 | Conclusiones y limitaciones |

---

> ### Nota sobre las Secciones 1 a 4
>
> Todo lo que sigue hasta los baselines de §4.3 está **copiado sin modificar** del notebook del
> Miniproyecto 1: mismo código, mismas gráficas, mismas lecturas. Es lo que garantiza que BETO
> se evalúe sobre exactamente la misma submuestra, el mismo Split A y con la misma función
> `evaluar(...)` que los modelos de las entregas anteriores.
>
> Sus textos hablan de los modelos de aquella entrega (TF-IDF, LSTM, BETO «excluido»…). **No es
> un descuido**: alterar el texto habría significado no reutilizar lo mismo. Al terminar el
> bloque, una celda puente retraduce cada hallazgo a las decisiones de este notebook. Algunas
> piezas heredadas (el vocabulario por palabras de §4.1, el Split B) no se usan aquí; se
> conservan para no romper la identidad del bloque.

---

## 1. Entorno y reproducibilidad

Antes de cualquier análisis fijamos las condiciones que hacen que este notebook produzca los
mismos números en cada ejecución, y que corra igual en Colab con GPU, en Colab sin GPU y en
una máquina local.

<!-- REDACTAR: por qué la reproducibilidad no es un trámite sino parte del resultado:
     sin semilla fija, comparar cuatro modelos no significa nada. -->

In [ ]:
import sys, os, warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Ejecutando en Colab: {IN_COLAB}')
print(f'Python: {sys.version.split()[0]}')

Instalamos únicamente lo que falte. En Colab casi todo viene preinstalado; el modelo de
vectores de spaCy en español es la descarga pesada (~570 MB) y solo se necesita a partir de
la sección 7.

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

x = torch.randn(3, 3, device="cuda" if torch.cuda.is_available() else "cpu")
print("prueba tensor OK:", (x @ x).shape)

In [ ]:
# =========================================================================
# 1 · Entorno y reproducibilidad
# =========================================================================
import sys, os, platform, random, warnings, time
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# --- ¿Estamos en Colab? ---
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# --- Dependencias que podrían faltar en Colab: vectores de spaCy y ftfy ---
import spacy
if not spacy.util.is_package("es_core_news_lg"):
    print("Descargando es_core_news_lg (~570 MB, 1-2 min)...")
    os.system(f"{sys.executable} -m spacy download es_core_news_lg")
    print("Descarga terminada.")

try:
    import ftfy  # noqa: F401
except ImportError:
    os.system(f"{sys.executable} -m pip install -q ftfy")

### Semilla global

Fijamos `SEED = 42` en todas las fuentes de aleatoriedad del pipeline: submuestreo,
particiones, inicialización de pesos y orden de los lotes.

In [ ]:
# --- Semilla global ---
import numpy as np
import torch

SEED = 42

def fijar_semilla(sem: int = SEED):
    """Fija la semilla en random, numpy y torch (incl. CUDA)."""
    random.seed(sem)
    np.random.seed(sem)
    torch.manual_seed(sem)
    torch.cuda.manual_seed_all(sem)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fijar_semilla()
G = torch.Generator().manual_seed(SEED)   # barajado reproducible de los DataLoader

fijar_semilla()
print("\nSemilla global fijada en", SEED)

### Configuración adaptativa

El notebook debe correr en cualquier entorno, no fallar en el que no tenga GPU. Esta celda
detecta el dispositivo y ajusta tamaños de lote, número de épocas y estrategia para el
transformer. La configuración activa se imprime para que quede registrada junto a los
resultados.

In [ ]:
# --- Dispositivo y configuración adaptativa ---
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HAY_GPU = DISPOSITIVO.type == "cuda"

if HAY_GPU:
    CFG = dict(submuestra=40_000, epocas=8, batch=128,
               emb_dim=128, hidden=128, max_vocab=30_000)
else:
    CFG = dict(submuestra=12_000, epocas=4, batch=64,
               emb_dim=96, hidden=96, max_vocab=25_000)
CFG.update(lr=2e-3, paciencia=3, seed=SEED)

# --- Traza del entorno (queda registrada en el .ipynb) ---
import sklearn, pandas as pd
try:
    import datasets
    ver_ds = datasets.__version__
except Exception:
    ver_ds = "no instalado"

print("Python      :", sys.version.split()[0], "| SO:", platform.system())
print("¿En Colab?  :", EN_COLAB)
print("Dispositivo :", DISPOSITIVO,
      f"({torch.cuda.get_device_name(0)})" if HAY_GPU else "")
print("Versiones   : torch", torch.__version__, "| sklearn", sklearn.__version__,
      "| pandas", pd.__version__, "| spacy", spacy.__version__, "| datasets", ver_ds)
print("\nCFG activa  :")
for k, v in CFG.items():
    print(f"  {k:11s}: {v}")

---

## 2. El corpus

Trabajamos con [`vg055/Rest-Mex2025`](https://huggingface.co/datasets/vg055/Rest-Mex2025):
208,051 reseñas turísticas en español sobre destinos de México, del shared task
**Rest-Mex 2025** de IberLEF. Licencia CC-BY-4.0.

**Por qué este corpus.** Rest-Mex 2025 además del texto y la polaridad (1–5), trae el pueblo, la región y el tipo de
establecimiento. Eso nos da tres cosas que otros corpus de sentimiento en español no ofrecen
a la vez: una **variable objetivo ordinal**, un **desbalance de clases real** (no fabricado)
y un **eje geográfico** con el que montar una prueba de generalización.

**Entrada de los modelos.** Concatenamos `Title` + `Review`. El título suele ser la señal
más condensada de la reseña («Todo excelente», «Nunca más»), y descartarlo sería desperdiciar información.

In [ ]:
# =========================================================================
# 2 · Carga del corpus Rest-Mex 2025
# =========================================================================
from datasets import load_dataset
from datetime import date

t0 = time.time()
ds = load_dataset("vg055/Rest-Mex2025", split="train")
df = ds.to_pandas()
print(f"Descarga + to_pandas: {time.time() - t0:.1f} s  |  fecha de descarga: {date.today().isoformat()}")

# --- Normalización mínima ---
df["Polarity"] = df["Polarity"].astype(int)
df["Title"]  = df["Title"].fillna("").astype(str).str.strip()
df["Review"] = df["Review"].fillna("").astype(str).str.strip()
df["texto"]  = (df["Title"] + ". " + df["Review"]).str.strip()   # entrada única de todos los modelos

# --- Inspección inicial ---
print("\nShape:", df.shape)
print("\nDtypes:")
print(df[["Title", "Review", "Polarity", "Town", "Region", "Type"]].dtypes.to_string())
display(df[["Title", "Review", "Polarity", "Town", "Region", "Type"]].head())

# --- Verificación contra docs/DATASET.md §3 (si no coincide, el corpus cambió en el Hub) ---
print("\n--- Verificación ---")
print(f"Registros          : {len(df):>8,}     (DATASET.md: 208,051)")
print(f"Regiones / Pueblos : {df['Region'].nunique():>3} / {df['Town'].nunique():<3}   (DATASET.md: 19 / 40)")
print(f"Tipos              : {sorted(df['Type'].unique())}")
print(f"Rango de Polarity  : {df['Polarity'].min()} - {df['Polarity'].max()}")
print("\nDistribución de Polarity (%):")
print((df["Polarity"].value_counts(normalize=True).sort_index() * 100).round(2).to_string())
print("\nDistribución de Type (%):")
print((df["Type"].value_counts(normalize=True) * 100).round(2).to_string())
print("\nNulos por columna:")
print(df[["Title", "Review", "Polarity", "Town", "Region", "Type"]].isna().sum().to_string())

Antes de cualquier estadística, conviene ver el material crudo. Estas tres reseñas (una negativa, una intermedia y una positiva) dan la medida del registro, la extensión y el tipo
de lenguaje al que nos enfrentamos.

In [ ]:
# --- Tres reseñas crudas, una por nivel de polaridad ---
fijar_semilla()
for estrellas in (1, 3, 5):
    r = df[df["Polarity"] == estrellas].sample(1, random_state=SEED).iloc[0]
    print("=" * 100)
    print(f"[{estrellas}★]   Región: {r['Region']}   |   Pueblo: {r['Town']}   |   Tipo: {r['Type']}")
    print(f"Título : {r['Title']}")
    cuerpo = r["Review"]
    print(f"Reseña : {cuerpo[:800]}{'...' if len(cuerpo) > 800 else ''}")
    print(f"(longitud real del cuerpo: {len(cuerpo)} caracteres)\n")

Lo que se observa en el material crudo:

- **Registro coloquial y reseñas multi-tema.** Una misma reseña cubre reserva, transporte,
  trato y comida (1★) o comida, vino, ubicación y organización (5★). El modelo debe integrar
  señales dispersas, no clasificar una sola frase.
- **La reseña de 3★ es explícitamente mixta:** «gran vista» / «la comida no es buena» /
  «la cerveza es barata». Elogio y queja conviven. Anticipamos que 3★ será la clase más
  difícil, y aquí se ve por qué.
- **Ruido de codificación:** *CancÃºn* en vez de *Cancún*, mojibake por doble codificación
  UTF-8/Latin-1 en el CSV de origen.
- **Marcadores de truncamiento de TripAdvisor:** reseñas que terminan en «...Más» o «….Más»,
  restos del botón «leer más». Ruido a limpiar antes de tokenizar.
- **Texto a veces poco natural** («El barco marchamos antes, lo siento»): parte del corpus
  está traducido automáticamente o escrito por hablantes no nativos. Es dificultad inherente
  a los datos, no del modelo.

<!-- REDACTAR: qué se observa en los ejemplos. Registro coloquial, mezcla de temas dentro
     de una misma reseña, anglicismos, uso de mayúsculas y signos. Anticipar aquí que la
     reseña de 3 estrellas probablemente contenga elogios y quejas mezclados: eso explicará
     buena parte de los errores de la sección 14. -->

---

## 3. Análisis exploratorio

Esta sección no es un requisito administrativo: de aquí salen cuatro decisiones concretas
la métrica principal, la longitud máxima de secuencia, el diseño de la partición
geográfica y el tratamiento de la ordinalidad, y todas quedan justificadas con datos en
§3.8.

El análisis se hace sobre **el corpus completo** (208k). Solo el entrenamiento usará una
submuestra.

### 3.1 Calidad de los datos

Los nulos ya sabemos que son cero. Aquí buscamos lo que sí puede
contaminar el análisis y el entrenamiento: **duplicados exactos** (la misma reseña repetida
caería a la vez en train y test, es decir data-leakage), reseñas demasiado cortas para contener sentimiento, y dos defectos de origen que ya vimos en crudo: el **mojibake** de codificación y los **marcadores `...Más`** de TripAdvisor.

Diagnosticamos primero, decidimos después, y aplicamos solo la limpieza que se pueda
justificar.

In [ ]:
# =========================================================================
# 3.1 · Diagnóstico de calidad
# =========================================================================
import re

def tiene_mojibake(s: str) -> bool:
    return ("Ã" in s) or ("Â¿" in s) or ("Â¡" in s) or ("Âº" in s)

_pat_mas = re.compile(r"(?:\.{2,}|…)\s*Más\s*$")

rep = {
    "registros"                     : len(df),
    "Title vacío"                   : int((df["Title"] == "").sum()),
    "Review vacío"                  : int((df["Review"] == "").sum()),
    "Review < 15 caracteres"        : int((df["Review"].str.len() < 15).sum()),
    "duplicados exactos de 'texto'" : int(df["texto"].duplicated().sum()),
    "duplicados (texto, Polarity)"  : int(df.duplicated(["texto", "Polarity"]).sum()),
    "texto con mojibake"            : int(df["texto"].map(tiene_mojibake).sum()),
    "Review termina en '...Más'"    : int(df["Review"].str.contains(_pat_mas).sum()),
    "Polarity fuera de 1..5"        : int((~df["Polarity"].between(1, 5)).sum()),
}
display(pd.DataFrame({"chequeo": list(rep), "valor": list(rep.values())}))

print("\nEjemplos de mojibake (Title):")
for s in df.loc[df["texto"].map(tiene_mojibake), "Title"].head(5).tolist():
    print("  ", repr(s))

print("\nEjemplos de reseñas muy cortas:")
for s in df.loc[df["Review"].str.len() < 15, "Review"].head(5).tolist():
    print("  ", repr(s))

if rep["duplicados exactos de 'texto'"]:
    d = df[df["texto"].duplicated(keep=False)].sort_values("texto")
    print("\nEjemplo de par duplicado:")
    for _, r in d.head(2).iterrows():
        print(f"  [{r['Polarity']}★] {r['texto'][:130]}")

In [ ]:
# =========================================================================
# 3.1 · Limpieza (solo lo justificable) — corpus de trabajo para todo el notebook
# =========================================================================
import ftfy   # instalado en la Sección 1 si hacía falta

n0 = len(df)

# 1) Duplicados exactos: se elimina la copia (evita fuga train/test).
df = df.drop_duplicates("texto").reset_index(drop=True)

# 2) Mojibake + marcadores '...Más'. NO se eliminan filas: se corrige el texto.
def limpiar_texto(s: str) -> str:
    s = ftfy.fix_text(s)               # repara doble codificación UTF-8/Latin-1
    s = _pat_mas.sub("", s)            # quita el '...Más' de cola de TripAdvisor
    return s.strip()

df["Title"]  = df["Title"].map(limpiar_texto)
df["Review"] = df["Review"].map(limpiar_texto)
df["texto"]  = (df["Title"] + ". " + df["Review"]).str.strip()

# 3) Las reseñas cortas NO se eliminan: son parte del dominio y la §14 (errores) las usa.

print(f"Duplicados exactos eliminados : {n0 - len(df):,}")
print(f"Corpus de trabajo            : {len(df):,} reseñas")
print(f"texto con mojibake tras limpiar: {int(df['texto'].map(tiene_mojibake).sum())}")
print(f"Review termina en '...Más' tras limpiar: {int(df['Review'].str.contains(_pat_mas).sum())}")
print("\nAntes / después (fila 3 del head original):")
print("  ", repr(df['Title'].iloc[3]))

In [ ]:
# =========================================================================
# 3.1 · Chequeos rápidos finales
# =========================================================================
# a) ¿Cada pueblo pertenece a una única región? (si no, sería error de datos)
pueblos_multi = (df.groupby("Town")["Region"].nunique() > 1).sum()
print("Pueblos asignados a más de una región:", int(pueblos_multi))

# b) ¿Están todas las reseñas en español? Heurística por palabras funcionales sobre muestra.
_es = {" el ", " la ", " los ", " las ", " de ", " que ", " y ", " un ", " una ", " con ", " para "}
_en = {" the ", " and ", " is ", " was ", " we ", " very ", " of ", " to ", " this ", " they "}
def idioma_aprox(t):
    t = " " + t.lower() + " "
    return "en" if sum(w in t for w in _en) > sum(w in t for w in _es) else "es"
muestra = df["texto"].sample(5000, random_state=SEED)
prop_en = (muestra.map(idioma_aprox) == "en").mean()
print(f"Reseñas que parecen inglés (muestra de 5.000): {prop_en*100:.1f}%")

# c) Colas largas: ¿hay reseñas monstruosamente largas que distorsionen el EDA de longitudes?
p = np.percentile(df["texto"].str.len(), [50, 95, 99, 99.9, 100]).astype(int)
print(f"\nLongitud de 'texto' (caracteres) — P50 {p[0]} | P95 {p[1]} | P99 {p[2]} | P99.9 {p[3]} | máx {p[4]}")
print("Reseñas con > 4.000 caracteres:", int((df["texto"].str.len() > 4000).sum()))

# d) Title vacío: ¿molesta? (el 'texto' arranca con '. ' en esos casos)
print("Title vacío tras limpieza:", int((df["Title"] == "").sum()))

**Diagnóstico.** El corpus está en muy buen estado. Sobre 208.051 reseñas:

- **182 duplicados exactos** (0.09 %), todos con la misma polaridad, no hay etiquetas
  contradictorias. Se eliminan para que una misma reseña no caiga a la vez en train y test.
- **4.885 reseñas con mojibake** (2.3 %) por doble codificación UTF-8/Latin-1 en el CSV de
  origen. `ftfy` corrige el 99.8 %; quedan 8 casos residuales, irrelevantes.
- **16.248 reseñas** (7.8 %) terminaban en el marcador «...Más» de TripAdvisor. Se recorta
  esa cola; el resto del texto no se toca.
- **0 reseñas vacías o degeneradas**, **0 polaridades fuera de rango**, cada pueblo pertenece
  a una única región, y solo ~0.1 % de las reseñas está en otro idioma: el corpus es
  genuinamente español, como se anuncia.
- Longitudes muy asimétricas (mediana 284 caracteres, P99 1.424, máximo 8.299; 61 reseñas
  sobre 4.000).

**Decisiones de limpieza:**

| Problema | Decisión | Por qué |
|---|---|---|
| Duplicados exactos | eliminar (−182) | fuga de información train/test |
| Mojibake | corregir con `ftfy` | ruido de origen, no señal |
| Marcador «...Más» | recortar la cola | artefacto de scraping |
| Reseñas cortas | **conservar** | son parte del dominio|
| Reseñas muy largas | **conservar** | se acotan con `MAX_LEN`|
| Título vacío (6 casos) | **conservar** | despreciable; el cuerpo está intacto |
| Texto traducido / poco natural | **conservar** | dificultad real del corpus, no un defecto |

### 3.2 Distribución de la polaridad

La polaridad es la etiqueta que queremos predecir: las estrellas (1 a 5) que el autor le
puso a su reseña. Antes de entrenar nada miramos cómo se reparten esas estrellas, porque de
ese reparto dependen tanto la métrica con la que mediremos el éxito como la forma de entrenar.

In [ ]:
# =========================================================================
# 3.2 · Distribución de la polaridad
# =========================================================================
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
# Paleta fija por estrella, usada en TODO el notebook (rojo = malo, azul = bueno).
PALETA_POL = {1: "#b2182b", 2: "#ef8a62", 3: "#fddbc7", 4: "#67a9cf", 5: "#2166ac"}

conteo = df["Polarity"].value_counts().sort_index()
pct = conteo / conteo.sum() * 100

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(conteo.index, conteo.values, color=[PALETA_POL[i] for i in conteo.index])
for b, n, p in zip(barras, conteo.values, pct):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(),
            f"{n:,}\n{p:.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Polaridad (estrellas)")
ax.set_ylabel("Número de reseñas")
ax.set_title("Distribución de la polaridad en el corpus (207.869 reseñas)")
ax.set_ylim(0, conteo.max() * 1.15)
plt.tight_layout()
plt.show()

print(pd.DataFrame({"n": conteo, "%": pct.round(2)}).to_string())
print(f"\nRatio mayoritaria/minoritaria : {conteo.max() / conteo.min():.1f} : 1")
print(f"Polaridad media / mediana     : {df['Polarity'].mean():.2f} / {df['Polarity'].median():.0f}")

In [ ]:
# =========================================================================
# 3.2 · Baseline de clase mayoritaria — de aquí sale la métrica principal
# =========================================================================
from sklearn.metrics import f1_score, accuracy_score

clase_mayoritaria = int(conteo.idxmax())
y_real = df["Polarity"].values
y_pred_may = np.full_like(y_real, clase_mayoritaria)

acc_may = accuracy_score(y_real, y_pred_may)
f1_may = f1_score(y_real, y_pred_may, average="macro", labels=[1, 2, 3, 4, 5], zero_division=0)
f1_por_clase = f1_score(y_real, y_pred_may, average=None, labels=[1, 2, 3, 4, 5], zero_division=0)

print(f"Baseline = predecir SIEMPRE {clase_mayoritaria}★ (sin leer el texto)")
print(f"  accuracy  = {acc_may:.4f}")
print(f"  macro-F1  = {f1_may:.4f}")
print(f"  F1 por clase 1..5 = {[round(v, 3) for v in f1_por_clase]}")

# Línea de referencia para TODAS las tablas de resultados.
BASELINE = {"clase": clase_mayoritaria, "accuracy": float(acc_may), "macro_f1": float(f1_may)}

La siguiente celda traduce ese desbalance en tres cosas concretas: cuánto pesará cada clase
al entrenar, si reformular la tarea ayudaría, y con cuántos ejemplos reales contaremos para
las clases poco frecuentes.

In [ ]:
# =========================================================================
# 3.2 (extra) · Qué implica el desbalance para el modelado
# =========================================================================
# a) Pesos de clase para la pérdida (inverso de la frecuencia, normalizado a media 1)
frec = conteo / conteo.sum()
w = (1 / frec) / (1 / frec).sum() * len(conteo)
print("class_weight (1..5):", {e: round(v, 2) for e, v in zip([1, 2, 3, 4, 5], w)})
print(f"  -> un error en 1★ pesará ~{w[1] / w[5]:.0f}x más que uno en 5★\n")

# b) ¿Se escapa del desbalance reformulando la tarea?
c3 = pd.Series(np.select([df["Polarity"] <= 2, df["Polarity"] == 3],
                         ["neg (1-2)", "neu (3)"], "pos (4-5)")).value_counts(normalize=True) * 100
c2 = pd.Series(np.where(df["Polarity"] >= 4, "pos (4-5)", "no-pos (1-3)")).value_counts(normalize=True) * 100
print("Colapsando a 3 clases (%):\n", c3.round(1).to_string())
print("\nColapsando a binario (%):\n", c2.round(1).to_string())

# c) Ejemplos de entrenamiento por clase tras la submuestra estratificada de 40k (split 80%)
proj = (frec * CFG["submuestra"] * 0.8).round().astype(int)
print(f"\nEjemplos de train por clase con submuestra de {CFG['submuestra']:,}:")
print({e: int(v) for e, v in zip([1, 2, 3, 4, 5], proj)})

# d) MAE del baseline mayoritario (para completar su fila en las tablas de resultados)
mae_may = float(np.abs(y_real - clase_mayoritaria).mean())
BASELINE["mae"] = mae_may
print(f"\nMAE del baseline 'siempre 5★': {mae_may:.3f} estrellas")

Dos de cada tres reseñas tienen 5 estrellas y solo cinco de cada cien tienen 1 o 2. Este sesgo hacia lo positivo es habitual en sitios de reseñas, así que no es un error a corregir sino una característica del problema.

La consecuencia práctica es que el accuracy (porcentaje de aciertos) deja de servir: un modelo que responda siempre "5 estrellas", sin leer el texto, acierta el 65,7 % de las
veces. Por eso medimos el éxito con el macro-F1, que evalúa la calidad clase por clase y luego promedia; ese mismo modelo perezoso saca un macro-F1 de 0,16, porque falla por
completo en cuatro de las cinco clases. Todo el notebook usa macro-F1 como criterio, y el accuracy solo se muestra al lado de este baseline como referencia.

Para que los modelos no ignoren las clases raras, al entrenar daremos a cada clase un peso inverso a su frecuencia, es decir, un error en una reseña de 1★ pesará unas 25 veces más que uno en
una de 5★.

Aun así el problema no desaparece: agrupando en positivo/neutro/negativo el corpus sigue siendo 87 % positivo, y al reducirlo a 40.000 reseñas para que todo corra en tiempo razonable quedarán apenas ~840 ejemplos de 1★ y ~845 de 2★ para entrenar. Es
suficiente para que el macro-F1 sea informativo, pero poco para conclusiones finas sobre esas dos clases; lo consideraremos en las limitaciones.

### 3.3 Tipo de establecimiento, región y pueblo

In [ ]:
# =========================================================================
# 3.3 · Tipo de establecimiento, región y pueblo
# =========================================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

vc_type = df["Type"].value_counts()
axes[0].bar(vc_type.index, vc_type.values, color="#4c72b0")
for i, v in enumerate(vc_type.values):
    axes[0].text(i, v, f"{v / len(df) * 100:.0f}%", ha="center", va="bottom")
axes[0].set_title("Type (3 clases)"); axes[0].set_ylabel("reseñas")

vc_reg = df["Region"].value_counts()
sns.barplot(x=vc_reg.head(12).values, y=vc_reg.head(12).index, ax=axes[1], color="#4c72b0")
axes[1].set_title(f"Region — top 12 de {df['Region'].nunique()}"); axes[1].set_xlabel("reseñas")

vc_town = df["Town"].value_counts()
sns.barplot(x=vc_town.head(12).values, y=vc_town.head(12).index, ax=axes[2], color="#4c72b0")
axes[2].set_title(f"Town — top 12 de {df['Town'].nunique()}"); axes[2].set_xlabel("reseñas")

plt.tight_layout(); plt.show()

print("Type (%):")
print((vc_type / len(df) * 100).round(1).to_string())
print("\nConcentración geográfica:")
print(f"  Quintana Roo         : {vc_reg.iloc[0] / len(df) * 100:.1f}%")
print(f"  top 3 regiones       : {vc_reg.head(3).sum() / len(df) * 100:.1f}%")
print(f"  Tulum + Isla Mujeres : {vc_town.head(2).sum() / len(df) * 100:.1f}%")
print("\nRegiones completas (n y %):")
print(pd.DataFrame({"n": vc_reg, "%": (vc_reg / len(df) * 100).round(2)}).to_string())

El tipo de establecimiento está bastante repartido: 41.7 % restaurantes, 33.6 % atracciones (museos, playas, zonas arqueológicas) y 24.7 % hoteles. Es lo contrario al caso de la polaridad. Por eso, nos puede convenir usar `Type` como tarea de control: si un mismo modelo clasifica bien
el tipo pero mal las estrellas, la dificultad está en la tarea de polaridad, no en el modelo.

La geografía, en cambio, está muy concentrada. Quintana Roo (Cancún, Tulum, Isla Mujeres, Bacalar) reúne el 41 % de las reseñas, las tres regiones más grandes suman el 62 %, y solo
Tulum e Isla Mujeres ya son el 36 %. De las 19 regiones, muchas aportan menos del 1 %.

Aqui hay un riesgo para el modelo, si entrenamos y evaluamos mezclando al azar, el modelo ve durante el entrenamiento reseñas de los mismos hoteles y playas que luego tiene que calificar, y podría aprender a reconocer el destino (nombres propios, topónimos) en lugar del sentimiento.

In [ ]:
# (opcional) polaridad media por tipo y por región — anticipo de 3.6
print("Polaridad media por Type:")
print(df.groupby("Type")["Polarity"].mean().round(2).to_string())
print("\nPolaridad media — regiones del held-out del Split B vs. resto:")
ho = ["Chiapas", "Baja_CaliforniaSur", "Queretaro"]
print(f"  held-out : {df[df.Region.isin(ho)]['Polarity'].mean():.2f}")
print(f"  resto    : {df[~df.Region.isin(ho)]['Polarity'].mean():.2f}")

Los hoteles reciben calificaciones algo más bajas que atracciones y restaurantes (4,32
frente a 4,52 y 4,47), aunque la diferencia es pequeña y todas las medias siguen muy altas.
Y las tres regiones que podríamos dejar fuera para una prueba de generalización tienen una
polaridad media (4,44) casi idéntica a la del resto (4,45): esa eventual partición no
arrancaría sesgada por el lado de las etiquetas. Queda como observación.

<!-- LEER: el contraste entre un `Type` balanceado y una polaridad que no lo está, de ahí
     sale la tarea de control de la §13, y la concentración geográfica en Quintana Roo,
     que motiva la partición por región de la §10. -->

### 3.4 Longitud de las reseñas → de dónde sale `MAX_LEN`

Los notebooks guía fijan la longitud de secuencia en 256 o 512 tokens sin justificarlo.
Aquí la derivamos de la distribución real: `MAX_LEN` será el percentil 95 en tokens, y
reportaremos qué fracción de reseñas queda truncada.

In [ ]:
# =========================================================================
# 3.4 · Longitud de las reseñas (caracteres y tokens) -> MAX_LEN
# =========================================================================
import re

# Tokenizador de trabajo solo para medir longitudes. El definitivo se fija en 4.1,
# pero usa el mismo criterio: minúsculas, se parte por lo que no sea letra o número,
# se conservan tildes y ñ.
_re_tok = re.compile(r"[^a-záéíóúüñ0-9]+")
def _tok(t):
    return [x for x in _re_tok.sub(" ", t.lower()).split() if x]

df["n_car"] = df["texto"].str.len()

# Contar tokens en las 207k reseñas es lento; usamos una muestra grande y reproducible.
muestra_tok = df["texto"].sample(60000, random_state=SEED).map(lambda t: len(_tok(t)))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].hist(df["n_car"].clip(upper=1500), bins=60, color="#4c72b0")
axes[0].set_title("Longitud en caracteres (recortada a 1500 para visualizar)")
axes[0].set_xlabel("caracteres"); axes[0].set_ylabel("reseñas")
axes[1].hist(muestra_tok.clip(upper=300), bins=60, color="#dd8452")
axes[1].set_title("Longitud en tokens (muestra de 60.000, recortada a 300)")
axes[1].set_xlabel("tokens"); axes[1].set_ylabel("reseñas")
plt.tight_layout(); plt.show()

pcts = [50, 75, 90, 95, 99]
tabla_long = pd.DataFrame({
    "percentil": pcts,
    "caracteres": np.percentile(df["n_car"], pcts).round().astype(int),
    "tokens": np.percentile(muestra_tok, pcts).round().astype(int),
})
print(tabla_long.to_string(index=False))

MAX_LEN = int(round(np.percentile(muestra_tok, 95) / 10) * 10)
trunc = (muestra_tok > MAX_LEN).mean()
print(f"\nMediana de tokens : {int(muestra_tok.median())}")
print(f"MAX_LEN (percentil 95 en tokens, redondeado a decena) = {MAX_LEN}")
print(f"Con ese límite se trunca el {trunc * 100:.1f}% de las reseñas (solo la cola larga).")

La mayoría de las reseñas son cortas: la mitad tiene 48 palabras o menos y el 90 % cabe en 136. Pero hay una cola larga que llega a más de 240. Fijamos el límite de longitud (`MAX_LEN`) en 150 palabras, que es el percentil 95: con eso solo se recorta el 4,4 % de las reseñas, y siempre por el final, que suele ser la parte menos informativa.

En los histogramas hay un pico alrededor de las 140 palabras (unos 800 caracteres) que parece un corte fijo del proceso de descarga de las reseñas, no algo propio del lenguaje. El pico del extremo derecho de cada gráfica es solo el efecto de amontonar ahí todo lo que se sale del rango dibujado; los valores reales están en la tabla de percentiles.

<!-- LEER: la asimetría de la distribución y por qué el P95 es el compromiso correcto. -->

### 3.5 Léxico distintivo por polaridad

Las frecuencias crudas están dominadas por palabras funcionales y no distinguen nada. Usamos
**log-odds ratio con prior de Dirichlet**, que mide qué tan sobrerrepresentado está un
término en una clase respecto al resto del corpus y es robusto ante términos poco frecuentes.

In [ ]:
# =========================================================================
# 3.5 · Léxico distintivo: 1★ frente a 5★ (log-odds con prior de Dirichlet)
# =========================================================================
from collections import Counter

def log_odds(textos_a, textos_b, alpha=0.01, top=18, min_freq=10):
    """z-score de log-odds (Monroe et al. 2008): qué términos distinguen A de B."""
    ca, cb = Counter(), Counter()
    for t in textos_a: ca.update(_tok(t))
    for t in textos_b: cb.update(_tok(t))
    vocab = set(ca) | set(cb)
    na, nb, a0 = sum(ca.values()), sum(cb.values()), alpha * len(vocab)
    z = {}
    for w in vocab:
        if ca[w] + cb[w] < min_freq:
            continue
        la = np.log((ca[w] + alpha) / (na + a0 - ca[w] - alpha))
        lb = np.log((cb[w] + alpha) / (nb + a0 - cb[w] - alpha))
        z[w] = (la - lb) / np.sqrt(1.0 / (ca[w] + alpha) + 1.0 / (cb[w] + alpha))
    ordenado = sorted(z.items(), key=lambda kv: kv[1])
    return ordenado[-top:][::-1], ordenado[:top]   # (más de A, más de B)

txt_1 = df.loc[df["Polarity"] == 1, "texto"]
txt_5 = df.loc[df["Polarity"] == 5, "texto"].sample(len(txt_1) * 2, random_state=SEED)
mas_1, mas_5 = log_odds(txt_1, txt_5)
LEXICO = {1: [w for w, _ in mas_1], 5: [w for w, _ in mas_5]}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, datos, titulo, color in [(axes[0], mas_1, "Más propio de 1★", "#b2182b"),
                                 (axes[1], mas_5, "Más propio de 5★", "#2166ac")]:
    palabras, scores = zip(*datos)
    ax.barh(range(len(palabras)), scores, color=color)
    ax.set_yticks(range(len(palabras))); ax.set_yticklabels(palabras); ax.invert_yaxis()
    ax.set_title(titulo); ax.set_xlabel("log-odds (z)")
plt.tight_layout(); plt.show()

print("LEXICO[1] (1★):", LEXICO[1])
print("LEXICO[5] (5★):", LEXICO[5])

En las reseñas de 1★ lo que más pesa es la negación: "no", "ni", "nada", "nunca". Es un hallazgo relevante en cuanto a elegir el modelo, porque un enfoque de bolsa de palabras (que cuenta términos sin mirar el orden) no puede distinguir "no recomiendo" de "recomiendo".
También aparecen mucho "dijeron", "dijo", "nos": en las reseñas malas la gente reconstruye conversaciones con el personal. Y "habitación" y "hotel" se inclinan al lado negativo, lo que encaja con que los hoteles tenían la media de estrellas más baja.

En 5★ dominan los adjetivos de entusiasmo ("excelente", "increíble", "hermoso", "amable", "agradable") mezclados con palabras de lugar ("playa", "ruinas", "lugar", "visita"). Esto
último conviene tenerlo en cuenta: parte de la señal no es sentimiento puro sino relacionado al tema (las atracciones reciben mejores calificaciones que los hoteles), así que un modelo podría acertar en parte fijándose en de qué habla la reseña y no en cómo la valora.

Guardamos ambas listas en `LEXICO` para tenerlas como referencia.

### 3.6 Interacciones

In [ ]:
# =========================================================================
# 3.6 · Interacciones: polaridad × tipo, × región, × longitud
# =========================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))

# 1) P(polaridad | tipo)
ct = pd.crosstab(df["Type"], df["Polarity"], normalize="index")
sns.heatmap(ct, annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("P(polaridad | tipo)"); axes[0].set_xlabel("polaridad"); axes[0].set_ylabel("")

# 2) P(polaridad | región) para las 8 regiones con más reseñas
top8 = df["Region"].value_counts().head(8).index
cr = pd.crosstab(df.loc[df["Region"].isin(top8), "Region"], df["Polarity"], normalize="index")
cr = cr.loc[top8]  # ordenadas por volumen
sns.heatmap(cr, annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=axes[1])
axes[1].set_title("P(polaridad | región)"); axes[1].set_xlabel("polaridad"); axes[1].set_ylabel("")

# 3) longitud (caracteres) por polaridad
sns.boxplot(data=df, x="Polarity", y="n_car", hue="Polarity", legend=False,
            palette=PALETA_POL, showfliers=False, ax=axes[2])
axes[2].set_title("Longitud (caracteres) por polaridad")
axes[2].set_xlabel("polaridad"); axes[2].set_ylabel("caracteres")

plt.tight_layout(); plt.show()

print("Longitud mediana (caracteres) por polaridad:")
print(df.groupby("Polarity")["n_car"].median().to_string())
print("\nPolaridad media por región (top 8), de menor a mayor:")
print(df[df["Region"].isin(top8)].groupby("Region")["Polarity"].mean().round(2).sort_values().to_string())

El sesgo positivo es bastante parejo. Por tipo, los hoteles concentran algo más de reseñas negativas (4 % de 1★ y 4 % de 2★, frente al 1–2 % de las atracciones) y algo menos de 5★, aunque la diferencia es moderada.

Por región, la proporción de 5★ va del 57 % (Jalisco) al 70 % (Quintana Roo), y las tres regiones que podríamos apartar para una prueba de generalización (Chiapas, Baja California Sur, Querétaro) caen dentro de ese rango: apartarlas no cambiaría de forma brusca la mezcla de etiquetas.

La longitud sí se relaciona con la polaridad: las reseñas negativas son más largas (mediana de ~340–360 caracteres en 1★ y 2★, frente a ~275 en 5★).
Esto tiene sentido, quien se queja justifica el enojo con detalle, mientras que un "todo excelente" se despacha en una frase.

<!-- LEER: ¿son los hoteles más criticados que las atracciones? ¿escriben más largo quienes
     se quejan? Si la longitud correlaciona con la polaridad, es una señal que el modelo
     puede explotar y que conviene tener presente al interpretar resultados. -->

### 3.7 Cobertura del vocabulario frente a los embeddings preentrenados

El modelo 3 inicializará sus embeddings con vectores de spaCy entrenados sobre español
general. Antes de hacerlo, medimos cuánto de nuestro vocabulario turístico y regional está
efectivamente cubierto.

In [ ]:
# =========================================================================
# 3.7 · Cobertura del vocabulario en los vectores de spaCy (es_core_news_lg)
# =========================================================================
import spacy

nlp_vec = spacy.load("es_core_news_lg")
print("Vectores en es_core_news_lg:", nlp_vec.vocab.vectors.shape, "(nº de palabras × dimensiones)")

# Vocabulario del corpus (muestra de 50.000 reseñas para que sea rápido)
cont_vocab = Counter()
for t in df["texto"].sample(50000, random_state=SEED):
    cont_vocab.update(_tok(t))

tipos = list(cont_vocab)
con_vector = [w for w in tipos if nlp_vec.vocab[w].has_vector]
cob_tipos = len(con_vector) / len(tipos)
cob_tokens = sum(cont_vocab[w] for w in con_vector) / sum(cont_vocab.values())

oov = [(w, n) for w, n in cont_vocab.most_common() if not nlp_vec.vocab[w].has_vector][:25]

print(f"\nPalabras distintas en el corpus (muestra): {len(tipos):,}")
print(f"Cobertura por TIPOS  (palabras distintas con vector): {cob_tipos * 100:.1f}%")
print(f"Cobertura por TOKENS (apariciones cubiertas)        : {cob_tokens * 100:.1f}%")
print("\nPalabras frecuentes SIN vector (OOV):")
for w, n in oov:
    print(f"  {w:22s} {n}")

Casi todas las palabras que aparecen en las reseñas (99,1 % de las apariciones) tienen vector preentrenado. La cobertura baja al 74 % si contamos palabras distintas, pero lo que
falta son casi en su totalidad nombres propios: destinos (Bacalar, Sayulita, Taxco, Chichén,
Teotihuacán) y nombres de hoteles y restaurantes (Ziggy, Rolandi, Nautibeach). Vocabulario
de sentimiento sin cubrir prácticamente no hay.

Esto tiene dos lecturas. Partir de vectores preentrenados es una buena apuesta, porque el modelo no tiene que aprender desde cero qué significan "excelente" o "pésimo". Y como los nombres de lugar no tienen vector, un modelo que los use tal cual no puede apoyarse en el destino para adivinar la nota; en cambio, si dejamos que ajuste los vectores durante el entrenamiento, podría construir representaciones para esos nombres y usarlas como atajo.

Por eso conviene probar las dos variantes, vectores congelados y vectores ajustados, y comparar.
También se nota algo de ruido ortográfico: "pátzcuaro" y "patzcuaro", "teotihuacán" y "teotihuacan", cuentan como palabras distintas por la tilde.

In [ ]:
# =========================================================================
# 3.7b · Tres cifras para cerrar el diagnóstico
# =========================================================================
# (1) ¿Por qué un vocabulario de 30.000? Cobertura de apariciones según el corte.
cont_full = Counter()
for t in df["texto"].sample(80000, random_state=SEED):
    cont_full.update(_tok(t))
total = sum(cont_full.values())
print("Cobertura de apariciones según el tamaño de vocabulario:")
for k in (10000, 20000, 30000, 50000):
    cub = sum(n for _, n in cont_full.most_common(k))
    print(f"  vocab {k:>6,} -> {cub / total * 100:.2f}%")
print(f"  (palabras distintas en la muestra: {len(cont_full):,})")

# (2) Ruido de etiqueta: texto que contradice las estrellas.
neg_fuerte = {"pésimo", "pesimo", "horrible", "terrible", "asqueroso", "sucio", "estafa",
              "engaño", "robo", "nunca", "jamás", "malísimo", "peor"}
pos_fuerte = {"excelente", "maravilloso", "hermoso", "increíble", "perfecto", "espectacular",
              "encantó", "encanto", "recomiendo"}
def contiene(texto, palabras):
    return len(set(_tok(texto)) & palabras) > 0

r5 = df.loc[df["Polarity"] == 5, "texto"].map(lambda t: contiene(t, neg_fuerte)).mean()
r1 = df.loc[df["Polarity"] == 1, "texto"].map(lambda t: contiene(t, pos_fuerte)).mean()
print(f"\nReseñas de 5★ con vocabulario muy negativo en el texto: {r5 * 100:.1f}%")
print(f"Reseñas de 1★ con vocabulario muy positivo en el texto: {r1 * 100:.1f}%")

# (3) Densidad de negación por polaridad (negaciones por cada 100 palabras).
neg_tok = {"no", "ni", "nada", "nunca", "tampoco", "jamás", "sin"}
def dens_neg(texto):
    toks = _tok(texto)
    return sum(w in neg_tok for w in toks) / max(len(toks), 1) * 100
print("\nNegaciones por cada 100 palabras, según la polaridad:")
for p in (1, 2, 3, 4, 5):
    s = df.loc[df["Polarity"] == p, "texto"].sample(
        min(5000, int((df["Polarity"] == p).sum())), random_state=SEED)
    print(f"  {p}★: {s.map(dens_neg).mean():.2f}")

### 3.8 Síntesis: del hallazgo a la decisión

<!-- REDACTAR - la celda más importante del EDA. Una lista numerada, cada línea con la
     forma «hallazgo → decisión → dónde se aplica»:

     1. Desbalance 65/22/7/3/3  →  macro-F1 como métrica principal  →  §4, todas las tablas
     2. Distribución de longitudes  →  MAX_LEN = P95  →  §4
     3. Concentración en Quintana Roo  →  partición geográfica  →  §10
     4. Objetivo ordinal  →  MAE y QWK + formulaciones ordinales  →  §9
     5. Cobertura de embeddings  →  expectativa sobre el modelo 3  →  §7
     6. `Type` balanceado  →  tarea de control  →  §13
-->

| Hallazgo del EDA | Decisión que induce |
|---|---|
| El corpus trae mojibake y colas "...Más" de TripAdvisor, pero cero reseñas vacías y solo 182 duplicados exactos. | Se corrige el texto (`ftfy` + recorte) y se eliminan los duplicados; nada más. Corpus de trabajo: 207.869 reseñas. |
| El 66 % de las reseñas son 5★; predecir siempre "5" da 65,7 % de accuracy pero 0,16 de macro-F1. | La métrica principal es **macro-F1**. El accuracy solo se reporta junto a este baseline. |
| Las clases 1★ y 2★ juntas son el 5 %; tras reducir a 40.000 reseñas quedan ~840 ejemplos de cada una. | Se pondera cada clase por el inverso de su frecuencia al entrenar, y se asume desde ya que las métricas de 1★ y 2★ serán ruidosas (limitación). |
| Longitudes muy asimétricas: mediana 48 palabras, percentil 95 = 148. | **`MAX_LEN` = 150 palabras** (percentil 95); trunca solo el 4 % de las reseñas. |
| En 1★ domina la negación: 3,75 por cada 100 palabras, frente a 1,11 en 5★. | Una bolsa de palabras no distingue "no recomiendo" de "recomiendo": se justifica probar modelos que miran el orden (LSTM) y en ambos sentidos (bidireccional). |
| El 99 % de las apariciones tiene vector preentrenado; lo que falta son nombres de destinos y de hoteles. | Se inicializan los embeddings con los de spaCy y se prueban **congelados y ajustados**, porque ajustar podría reintroducir el destino como atajo. |
| Quintana Roo es el 41 % del corpus; con partición aleatoria el modelo ve en entrenamiento los mismos destinos que evalúa. | Se prueba además una **partición geográfica** (Chiapas, Baja California Sur y Querétaro fuera del entrenamiento) para medir generalización a destinos nuevos. |
| La escala 1–5 está ordenada: confundir 4★ con 5★ no equivale a confundir 1★ con 5★. | Además de macro-F1 se reportan **MAE y QWK**, y se prueba una formulación de regresión con umbrales. |
| `Type` está equilibrado (42/34/25) y no es ordinal. | Sirve de **tarea de control**: la misma arquitectura sobre `Type` dirá si un macro-F1 flojo en polaridad es culpa de la tarea o del modelo. |
| El 15 % de las reseñas de 1★ usan vocabulario muy positivo (y el 3 % de las de 5★, muy negativo). | Hay ruido de etiqueta y reseñas mixtas; ponen un techo al rendimiento y se analizan en la sección de errores. |

---

## 4. Preprocesamiento y protocolo experimental

Para que la comparación entre cuatro modelos signifique algo, todos deben ver exactamente
los mismos datos, la misma partición y las mismas métricas. Esta sección fija ese contrato.

### 4.1 Tokenización

Partimos del tokenizador del notebook guía y corregimos dos cosas: construimos el vocabulario
**por frecuencia** (el notebook 4 lo hace por orden de aparición, ver `docs/DECISIONS.md`
§D-002) y verificamos que la normalización preserve tildes y ñ.

In [ ]:
# =========================================================================
# 4.1 · Tokenizador definitivo y construcción del vocabulario
# =========================================================================
_RE_TOKEN = re.compile(r"[^a-záéíóúüñ0-9]+")

def tokenizar(texto: str):
    """Minúsculas; se parte por todo lo que no sea letra o número; se conservan
    tildes y ñ. Mismo criterio que se usó en el EDA."""
    return [t for t in _RE_TOKEN.sub(" ", texto.lower()).split() if t]

PAD, UNK = 0, 1

def construir_vocab(textos, max_vocab):
    """Vocabulario por FRECUENCIA (most_common), con [PAD]=0 y [UNK]=1 reservados.
    El notebook 4 del curso lo construye por orden de aparición: sobre un Counter
    eso no da los tokens más frecuentes sino los que aparecieron primero."""
    cont = Counter()
    for t in textos:
        cont.update(tokenizar(t))
    vocab = {"[PAD]": PAD, "[UNK]": UNK}
    for palabra, _ in cont.most_common(max_vocab - 2):
        vocab[palabra] = len(vocab)
    return vocab

ej = df["texto"].iloc[0]
print("Reseña :", ej[:150], "...")
print("Tokens :", tokenizar(ej)[:25], "...")
print("\nEl vocabulario definitivo se construye en 4.2, solo con el texto de entrenamiento.")

In [ ]:
# --- Cómo se convierte una reseña en entrada para el modelo: tokens -> ids -> padding ---
def codificar(texto, vocab, max_len=MAX_LEN):
    ids = [vocab.get(t, UNK) for t in tokenizar(texto)[:max_len]]
    n_real = max(len(ids), 1)                 # longitud antes del relleno
    ids += [PAD] * (max_len - len(ids))
    return ids, n_real

# Vocabulario temporal (solo para esta demostración; el real se hace en 4.2)
_vocab_demo = construir_vocab(df["texto"], CFG["max_vocab"])
_id2tok = {i: t for t, i in _vocab_demo.items()}

frase = "El hotel estaba sucio y el sargazo cubría toda la playa. No volvería."
ids, n = codificar(frase, _vocab_demo, max_len=20)
print("Texto        :", frase)
print("Tokens       :", tokenizar(frase))
print("IDs          :", ids)
print("Reconstruido :", [_id2tok[i] for i in ids])
print(f"\n{n} tokens reales; los otros {20 - n} son [PAD] (relleno para que todas las")
print("secuencias tengan el mismo largo y se puedan procesar por lotes).")
del _vocab_demo, _id2tok

El tokenizador parte el texto en palabras, pasa todo a minúsculas y elimina signos y saltos
de línea, pero conserva tildes y ñ (importante en español). Cada palabra se sustituye por un
número, que es su posición en el vocabulario; las palabras que no estén en el vocabulario se
marcan como `[UNK]`, y las secuencias más cortas que `MAX_LEN` se completan con `[PAD]` para
que todos los ejemplos tengan el mismo largo y se puedan procesar por lotes.

El vocabulario se ordena por frecuencia: las palabras más comunes ("el", "y", "la") reciben
los números más bajos y las raras ("cubría"), los más altos. En el ejemplo, las 13 palabras
reales caben de sobra en el límite de 150 y ninguna quedó fuera del vocabulario.

<!-- LEER: tamaño del vocabulario, tasa de UNK, qué se pierde al truncar. -->

### 4.2 Submuestra y particiones

Entrenamos sobre 40,000 reseñas estratificadas por polaridad. **No balanceamos las clases**:
el desbalance es una propiedad del dominio que queremos estudiar, no un defecto que ocultar.

In [ ]:
# =========================================================================
# 4.2 · Submuestra estratificada
# =========================================================================
from sklearn.model_selection import train_test_split

N_SUB = min(CFG["submuestra"], len(df))
sub, _ = train_test_split(df, train_size=N_SUB, random_state=SEED, stratify=df["Polarity"])
sub = sub.reset_index(drop=True)

comp = pd.DataFrame({
    "corpus (%)":     (df["Polarity"].value_counts(normalize=True).sort_index() * 100).round(2),
    "submuestra (%)": (sub["Polarity"].value_counts(normalize=True).sort_index() * 100).round(2),
    "n en submuestra": sub["Polarity"].value_counts().sort_index(),
})
print(f"Submuestra: {len(sub):,} reseñas (de {len(df):,}), estratificada por polaridad.")
print(comp.to_string())

La submuestra de 40.000 reseñas conserva exactamente las proporciones del corpus (66 % de
5★, 2,6 % de 1★).

In [ ]:
# =========================================================================
# 4.2 · Split A — aleatorio estratificado 80/10/10 (protocolo estándar)
# =========================================================================
idx = np.arange(len(sub))
y = sub["Polarity"].values

idx_tr, idx_tmp = train_test_split(idx, test_size=0.20, random_state=SEED, stratify=y)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, random_state=SEED, stratify=y[idx_tmp])
SPLIT_A = {"train": idx_tr, "val": idx_va, "test": idx_te}

print("Split A:")
for k, v in SPLIT_A.items():
    d = (pd.Series(y[v]).value_counts(normalize=True).sort_index() * 100).round(1)
    print(f"  {k:5s}: {len(v):>6,} reseñas  |  % por estrella: {d.tolist()}")

# Vocabulario definitivo: SOLO con el texto de entrenamiento de Split A (sin fuga)
vocab = construir_vocab(sub.iloc[SPLIT_A["train"]]["texto"], CFG["max_vocab"])
id2tok = {i: t for t, i in vocab.items()}
print(f"\nVocabulario definitivo: {len(vocab):,} tokens")

# Tasa de [UNK] y de truncamiento sobre el conjunto de test
unk = tot = trunc = 0
for t in sub.iloc[SPLIT_A["test"]]["texto"]:
    toks = tokenizar(t)
    tot += len(toks); unk += sum(w not in vocab for w in toks); trunc += len(toks) > MAX_LEN
print(f"En test -> tokens fuera del vocabulario: {unk / tot * 100:.1f}%  |  "
      f"reseñas truncadas por MAX_LEN: {trunc / len(SPLIT_A['test']) * 100:.1f}%")

El Split A la divide en 80/10/10 manteniendo esas proporciones en las tres partes, así que entrenamiento, validación y prueba son comparables entre sí. El vocabulario definitivo (30.000 palabras, construido solo con el texto de entrenamiento) deja fuera apenas el 1,5 % de las palabras del test, y el límite de 150 palabras trunca el 4,2 %, es decir, perdidas pequeñas.

In [ ]:
# =========================================================================
# 4.2 · Split B — geográfico (regiones fuera del entrenamiento)
# =========================================================================
REGIONES_FUERA = ["Chiapas", "Baja_CaliforniaSur", "Queretaro"]

es_fuera = sub["Region"].isin(REGIONES_FUERA).values
idx_b_test = idx[es_fuera]
idx_b_rest = idx[~es_fuera]
idx_b_tr, idx_b_va = train_test_split(idx_b_rest, test_size=0.12, random_state=SEED,
                                      stratify=y[idx_b_rest])
SPLIT_B = {"train": idx_b_tr, "val": idx_b_va, "test": idx_b_test}

print(f"Regiones fuera del entrenamiento: {REGIONES_FUERA}")
for k, v in SPLIT_B.items():
    print(f"  {k:5s}: {len(v):>6,} reseñas")

print("\nComprobación de que el test geográfico no está sesgado:")
print(f"  polaridad media : {sub.iloc[idx_b_test]['Polarity'].mean():.2f}  vs  "
      f"{sub.iloc[idx_b_rest]['Polarity'].mean():.2f}  (resto)")
print(f"  % hoteles       : {(sub.iloc[idx_b_test]['Type'] == 'Hotel').mean() * 100:.0f}%  vs  "
      f"{(sub.iloc[idx_b_rest]['Type'] == 'Hotel').mean() * 100:.0f}%  (resto)")
print(f"  % 5★            : {(sub.iloc[idx_b_test]['Polarity'] == 5).mean() * 100:.0f}%  vs  "
      f"{(sub.iloc[idx_b_rest]['Polarity'] == 5).mean() * 100:.0f}%  (resto)")

El Split B aparta tres regiones enteras (Chiapas, Baja California Sur y Querétaro; 7.456 reseñas, el 18,5 %). La polaridad media se ve casi igual (4,43 vs 4,46) y la proporción de 5★ es casi igual (64 % vs 66 %). Tiene algo más de hoteles (28 % vs 24 %), que suelen recibir peores notas, así que si el rendimiento cae en el Split B habrá que descontar esa pequeña diferencia.

### 4.3 Métricas y baselines

Una única función de evaluación que devuelve las seis métricas, usada sin excepción por
todos los modelos del notebook. Es lo que hace que la tabla final sea comparable.

In [ ]:
# =========================================================================
# 4.3 · Función de evaluación única — todos los modelos la usan sin excepción
# =========================================================================
from sklearn.metrics import (f1_score, accuracy_score, mean_absolute_error,
                             cohen_kappa_score, confusion_matrix, classification_report)

ESTRELLAS = [1, 2, 3, 4, 5]
RESULTADOS = []   # una fila por modelo
PREDS = {}        # nombre -> (y_true, y_pred) en el test de Split A, para el análisis de errores

def evaluar(y_true, y_pred, nombre, registrar=True, n_params=None, segundos=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    f1c = f1_score(y_true, y_pred, average=None, labels=ESTRELLAS, zero_division=0)
    fila = {
        "modelo":   nombre,
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=ESTRELLAS, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "mae":      mean_absolute_error(y_true, y_pred),
        "qwk":      cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=ESTRELLAS),
        **{f"f1_{e}": v for e, v in zip(ESTRELLAS, f1c)},
        "n_params": n_params, "segundos": segundos,
    }
    if registrar:
        RESULTADOS.append(fila)
    print(f"{nombre:38s} | macro-F1 {fila['macro_f1']:.4f} | acc {fila['accuracy']:.4f} "
          f"| MAE {fila['mae']:.3f} | QWK {fila['qwk']:.4f}")
    return fila

print("evaluar() lista. Métricas que devuelve:")
print("  macro-F1 : media de la F1 de las 5 clases (métrica principal del notebook)")
print("  accuracy : % de aciertos (solo se lee junto al baseline mayoritario)")
print("  MAE      : error medio en estrellas (0.5 = se equivoca media estrella de media)")
print("  QWK      : acuerdo con la nota real penalizando más los errores grandes (1=perfecto, 0=azar)")
print("  f1_1..f1_5 : F1 de cada clase por separado")

In [ ]:
# =========================================================================
# 4.3 · Baselines — ningún modelo es "útil" si no los supera en macro-F1
# =========================================================================
y_tr_A = sub.iloc[SPLIT_A["train"]]["Polarity"].values
y_te_A = sub.iloc[SPLIT_A["test"]]["Polarity"].values

# 1) Clase mayoritaria: predecir siempre la estrella más frecuente del entrenamiento
clase = int(pd.Series(y_tr_A).value_counts().idxmax())
evaluar(y_te_A, np.full_like(y_te_A, clase), f"Baseline · siempre {clase}★")

# 2) Azar estratificado: sortear según la distribución del entrenamiento
fijar_semilla()
p = pd.Series(y_tr_A).value_counts(normalize=True).sort_index()
y_azar = np.random.choice(p.index.values, size=len(y_te_A), p=p.values)
evaluar(y_te_A, y_azar, "Baseline · azar estratificado")

pd.DataFrame(RESULTADOS)[["modelo", "macro_f1", "accuracy", "mae", "qwk"]].round(4)

Los dos baselines dejan claro por qué hacen falta varias métricas a la vez. Predecir siempre "5★" tiene buen accuracy (0,66) pero macro-F1 de 0,16, porque cuatro clases quedan en cero.
Sortear al azar respetando las proporciones sube el macro-F1 a 0 19 (todas las clases reciben algo de acierto) pero hunde el accuracy a 0,48 y empeora el error medio a 0,83 estrellas. Y el QWK de ambos es prácticamente cero: ninguno tiene el más mínimo acuerdo real con la nota que puso el autor.

Un modelo solo cuenta como útil si supera el macro-F1 del azar (0,19) sin caer por debajo del accuracy de la clase mayoritaria (0,66), y si además consigue un QWK claramente positivo.

---

> ### Fin del bloque heredado: del EDA a esta entrega
>
> Aquí termina lo copiado del Miniproyecto 1. La tabla de §3.8 mapea hallazgos a las secciones
> de aquel notebook; esta los retraduce a las decisiones de este.

| Hallazgo del EDA | Qué decide en esta entrega | Dónde |
|---|---|---|
| 66 % de 5★: la mayoritaria da 65,7 % de accuracy y 0,16 de macro-F1 | **macro-F1** como métrica principal. El guía reporta solo accuracy | §4.3 (heredada) |
| 1★ y 2★ con ~840 ejemplos cada una en la submuestra | Pérdida ponderada por clase también en el `Trainer` (el guía no pondera) | §7 |
| Mediana 48 palabras, P95 = 148 | `MAX_LEN_BERT` = P95 **en tokens WordPiece**, no los 512 del guía ni las 150 palabras de MP1; padding dinámico | §4.6 |
| 26 % de las palabras distintas sin vector en spaCy | WordPiece no tiene `[UNK]` de palabra completa: fragmenta. Se mide la fertilidad | §4.5 |
| En 1★ domina la negación | Pruebas de estrés dirigidas e *Integrated Gradients* sobre reseñas con negación | §14, §15 |
| `Type` balanceado frente a polaridad desbalanceada | Tarea de control | §13 |

<!-- LEER: revisar la tabla tras ejecutar; ajustar cifras si el EDA cambió. -->

In [ ]:
# --- Verificación: el bloque heredado es idéntico al del Miniproyecto 1 ---
# Se ejecuta solo si el notebook de MP1 está disponible (en local); en Colab se omite.
import json, pathlib
_ruta_mp1 = pathlib.Path("../../miniproyecto 1/notebooks/miniproyecto1_restmex.ipynb")
if _ruta_mp1.exists():
    _mp1 = json.loads(_ruta_mp1.read_text(encoding="utf8"))["cells"][3:71]
    _nb = json.loads(pathlib.Path("miniproyecto3_restmex_bert.ipynb").read_text(encoding="utf8"))["cells"]
    _her = [c for c in _nb if "heredado-mp1" in c.get("metadata", {}).get("tags", [])]
    _txt = lambda c: "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
    assert len(_her) == len(_mp1) and all(_txt(a) == _txt(b) for a, b in zip(_her, _mp1))
    print(f"OK: las {len(_her)} celdas heredadas son idénticas a las del Miniproyecto 1.")
else:
    print("Notebook de MP1 no disponible en este entorno; verificación omitida.")

Comprobamos que el protocolo heredado se reprodujo exactamente: si los baselines no dan los
mismos números que en el Miniproyecto 1, el split no es el mismo y ninguna comparación entre
entregas sería válida.

In [ ]:
# --- Bloqueante: los baselines heredados reproducen los del Miniproyecto 1 ---
if CFG["submuestra"] == 40_000:
    assert abs(RESULTADOS[0]["accuracy"] - 0.6565) < 1e-3, "El split no reproduce el de MP1"
    assert abs(RESULTADOS[0]["macro_f1"] - 0.1585) < 1e-3, "El split no reproduce el de MP1"
    print("OK: submuestra y Split A idénticos a los del Miniproyecto 1.")
else:
    print("Configuración degradada (sin GPU): la comparación con MP1 no aplica.")

### 4.6 Dependencias y configuración propias de BERT

La configuración heredada (`CFG`) es la de los modelos del Miniproyecto 1 y se deja intacta. BERT
necesita la suya: `CFG_BERT` para las corridas principales y `CFG_EST`, reducida, para los
estudios de las Secciones 9–13 (`docs/DECISIONS.md` §D-306). Sin GPU el notebook no falla:
reduce épocas y longitud y lo anuncia.

In [ ]:
# =========================================================================
# 4.6 · Dependencias y configuración de BERT
# =========================================================================
def _asegurar(modulo, paquete=None):
    try:
        __import__(modulo)
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q {paquete or modulo}")

for m, p in [("transformers", None), ("accelerate", None), ("torchinfo", None),
             ("peft", None), ("captum", None), ("umap", "umap-learn")]:
    _asegurar(m, p)

import transformers
from transformers import set_seed

CHECKPOINT = "dccuchile/bert-base-spanish-wwm-cased"   # el del notebook guía

if HAY_GPU:
    CFG_BERT = dict(epocas=3, batch=32, batch_eval=128, max_len_tope=256)
    CFG_EST = dict(n_train=8_000, epocas=2)
else:
    print("⚠ Sin GPU: configuración degradada (resultados NO comparables con la corrida de referencia).")
    CFG_BERT = dict(epocas=1, batch=16, batch_eval=64, max_len_tope=128)
    CFG_EST = dict(n_train=800, epocas=1)

# lr 2e-5: aquí SÍ es la tasa correcta (fine-tuning de un preentrenado). En MP2 §D-204 era la
# trampa, porque allí se entrenaba desde cero.
CFG_BERT.update(lr=2e-5, lr_cabeza=1e-3, warmup_frac=0.10, weight_decay=0.01, paciencia=2,
                fp16=HAY_GPU, seed=SEED)

def fijar_semilla_hf(sem: int = SEED):
    # fijar_semilla (heredada) + la semilla interna de transformers
    fijar_semilla(sem); set_seed(sem)

print("transformers:", transformers.__version__, "| checkpoint:", CHECKPOINT)
for k, v in {**CFG_BERT, **{f"est.{k}": v for k, v in CFG_EST.items()}}.items():
    print(f"  {k:16s}: {v}")

### 4.5 El tokenizador de BETO frente a este corpus

WordPiece *cased* con 31.002 subpalabras, entrenado sobre texto formal en español. Nuestras
reseñas son coloquiales, con mexicanismos y nombres de destinos. El EDA §3.7 midió que el 26 %
de las palabras distintas no tenía vector en spaCy; aquí esas palabras no desaparecen en
`[UNK]`: se **fragmentan**. Medimos cuánto (fertilidad = subpalabras por palabra) y qué
palabras se rompen más.

In [ ]:
# =========================================================================
# 4.5 · Análisis del tokenizador WordPiece
# =========================================================================
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def fertilidad(tok, textos):
    """Subpalabras por palabra (separando por espacios) y tasa de [UNK]."""
    n_pal = n_sub = n_unk = 0
    for t in textos:
        ids = tok(t, add_special_tokens=False)["input_ids"]
        n_pal += len(t.split()); n_sub += len(ids); n_unk += sum(i == tok.unk_token_id for i in ids)
    return n_sub / max(n_pal, 1), n_unk / max(n_sub, 1)

import contextlib, io

# y_tr_A / y_te_A vienen del bloque heredado (§4.3); la validación la necesitamos aquí.
y_va_A = sub.iloc[SPLIT_A["val"]]["Polarity"].values

def evaluar_silencioso(y_true, y_pred):
    """La misma evaluar() heredada, sin registrar ni imprimir (early stopping por época)."""
    with contextlib.redirect_stdout(io.StringIO()):
        return evaluar(y_true, y_pred, "val", registrar=False)

txt_tr = sub.iloc[SPLIT_A["train"]]["texto"].tolist()
txt_va = sub.iloc[SPLIT_A["val"]]["texto"].tolist()
txt_te = sub.iloc[SPLIT_A["test"]]["texto"].tolist()

fert, unk = fertilidad(tokenizer, txt_tr[:5000])
print(f"Vocabulario: {tokenizer.vocab_size:,} | fertilidad: {fert:.2f} subpalabras/palabra | [UNK]: {unk*100:.2f}%")

for frase in ["El cenote de Tulum estaba padrísimo, pero el hostal chafa.",
              "No volvería: sargazo por todos lados en Mahahual."]:
    print("\n", frase, "\n ->", tokenizer.tokenize(frase))
# TODO(fase 2): top-20 palabras más fragmentadas del corpus y su frecuencia

<!-- LEER: ¿qué se fragmenta más? ¿es coherente con §3.7? -->

### 4.6 `MAX_LEN_BERT`: el P95 en tokens, no 512

El guía fija 512 porque sus noticias tienen mediana ~500 palabras. Aquí la mediana es 48. Con
atención cuadrática, rellenar a 512 multiplica el costo sin aportar información. Tomamos el P95
de la longitud en tokens WordPiece del train, y además usamos **padding dinámico**: cada lote se
rellena solo hasta su secuencia más larga.

In [ ]:
# =========================================================================
# 4.6 · MAX_LEN_BERT = P95 en tokens WordPiece (train)
# =========================================================================
import matplotlib.pyplot as plt, seaborn as sns

long_tr = np.array([len(i) for i in tokenizer(txt_tr, add_special_tokens=True)["input_ids"]])
p95 = int(np.percentile(long_tr, 95))
MAX_LEN_BERT = int(min(CFG_BERT["max_len_tope"], int(np.ceil(p95 / 8) * 8)))
long_te = np.array([len(i) for i in tokenizer(txt_te, add_special_tokens=True)["input_ids"]])

print(f"Mediana: {np.median(long_tr):.0f} tokens | P95: {p95} -> MAX_LEN_BERT = {MAX_LEN_BERT}")
print(f"Reseñas truncadas en test: {(long_te > MAX_LEN_BERT).mean()*100:.1f}%")
print(f"Costo relativo de la atención frente a 512: ~{(MAX_LEN_BERT/512)**2*100:.0f}% (sin contar padding dinámico)")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(long_tr, bins=80, color="#4C72B0")
for x, lab in [(MAX_LEN_BERT, f"MAX_LEN_BERT={MAX_LEN_BERT}"), (512, "guía: 512")]:
    ax.axvline(x, ls="--", color="k"); ax.text(x, ax.get_ylim()[1]*0.9, " " + lab)
ax.set_xlabel("longitud en tokens WordPiece"); ax.set_ylabel("reseñas"); ax.set_title("Longitud tokenizada (train)")
plt.tight_layout(); plt.show()

<!-- LEER: tasa de truncamiento y ahorro frente a 512. -->

### 4.7 Conjuntos para Hugging Face y pesos de clase

Etiquetas internas 0–4 (el modelo las necesita así); `evaluar` recibe siempre estrellas 1–5.

In [ ]:
# =========================================================================
# 4.7 · DatasetDict tokenizado + pesos de clase
# =========================================================================
from datasets import Dataset, DatasetDict
from transformers import DataCollatorWithPadding

def a_dataset(textos, estrellas):
    return Dataset.from_dict({"text": list(textos), "label": (np.asarray(estrellas) - 1).tolist()})

ds = DatasetDict(train=a_dataset(txt_tr, y_tr_A), val=a_dataset(txt_va, y_va_A), test=a_dataset(txt_te, y_te_A))

def tokenizar_ds(tok, max_len=None):
    return lambda b: tok(b["text"], truncation=True, max_length=max_len or MAX_LEN_BERT)

ds_tok = ds.map(tokenizar_ds(tokenizer), batched=True, remove_columns=["text"])
collator = DataCollatorWithPadding(tokenizer)

conteo = np.bincount(ds["train"]["label"], minlength=5)
PESOS = torch.tensor(len(ds["train"]) / (5 * conteo), dtype=torch.float)
print(ds_tok); print("Pesos de clase:", PESOS.numpy().round(2))

---

## 5. Técnica A — BETO congelado + cabeza lineal

Como en el guía, BETO se usa como **extractor de características** y solo se entrena una capa
lineal. Pero el guía vuelve a pasar cada reseña por los 110 M de parámetros congelados en cada
época. Si BETO no cambia, su salida tampoco: **la calculamos una sola vez** y entrenamos la
cabeza sobre el tensor cacheado. Mismo modelo, una fracción del tiempo.

Comparamos dos formas de resumir la secuencia: el token `[CLS]` (lo que usa
`BertForSequenceClassification`) y el **promedio enmascarado** de todos los tokens (la decisión
de MP2 §D-205). Con BERT preentrenado solo con MLM, el `[CLS]` no tiene por qué estar
organizado para nada en particular.

In [ ]:
# =========================================================================
# 5 · Extracción de embeddings congelados (una sola vez)
# =========================================================================
from transformers import AutoModel

@torch.no_grad()
def extraer_embeddings(modelo, dataset_tok, batch=CFG_BERT["batch_eval"]):
    """Devuelve (cls, mean) para todo el conjunto. BETO en modo eval, sin gradiente."""
    modelo.eval().to(DISPOSITIVO)
    dl = torch.utils.data.DataLoader(dataset_tok.remove_columns(["label"]), batch_size=batch, collate_fn=collator)
    cls, mean = [], []
    for b in dl:
        b = {k: v.to(DISPOSITIVO) for k, v in b.items()}
        with torch.autocast(DISPOSITIVO.type, enabled=HAY_GPU):
            h = modelo(**b).last_hidden_state.float()
        m = b["attention_mask"].unsqueeze(-1).float()
        cls.append(h[:, 0].cpu()); mean.append(((h * m).sum(1) / m.sum(1)).cpu())
    return torch.cat(cls), torch.cat(mean)

t0 = time.time()
beto = AutoModel.from_pretrained(CHECKPOINT)
EMB = {split: extraer_embeddings(beto, ds_tok[split]) for split in ["train", "val", "test"]}
SEG_EXTRACCION = time.time() - t0
N_PARAMS_BETO = sum(p.numel() for p in beto.parameters())
print(f"Embeddings extraídos en {SEG_EXTRACCION:.0f} s | BETO: {N_PARAMS_BETO/1e6:.1f} M parámetros")
del beto; torch.cuda.empty_cache()

Entrenamos la cabeza con el mismo bucle para todas las variantes: pérdida ponderada, AdamW y
early stopping por macro-F1 de validación.

In [ ]:
# =========================================================================
# 5 · Bucle de entrenamiento de cabezas sobre embeddings cacheados
# =========================================================================
import torch.nn as nn

def entrenar_cabeza(cabeza, Xtr, ytr, Xva, yva, epocas=30, lr=CFG_BERT["lr_cabeza"], batch=256, paciencia=5):
    fijar_semilla_hf()
    cabeza = cabeza.to(DISPOSITIVO)
    opt = torch.optim.AdamW(cabeza.parameters(), lr=lr, weight_decay=1e-2)
    crit = nn.CrossEntropyLoss(weight=PESOS.to(DISPOSITIVO))
    ytr_t = torch.tensor(ytr - 1); mejor, estado, sin_mejora = -1, None, 0
    t0 = time.time()
    for ep in range(epocas):
        cabeza.train()
        for i in torch.randperm(len(Xtr)).split(batch):
            opt.zero_grad()
            crit(cabeza(Xtr[i].to(DISPOSITIVO)), ytr_t[i].to(DISPOSITIVO)).backward(); opt.step()
        f1 = evaluar_silencioso(yva, predecir_cabeza(cabeza, Xva))["macro_f1"]
        if f1 > mejor: mejor, estado, sin_mejora = f1, {k: v.clone() for k, v in cabeza.state_dict().items()}, 0
        else:
            sin_mejora += 1
            if sin_mejora >= paciencia: break
    cabeza.load_state_dict(estado)
    return cabeza, time.time() - t0

@torch.no_grad()
def predecir_cabeza(cabeza, X):
    cabeza.eval(); return cabeza(X.to(DISPOSITIVO)).argmax(-1).cpu().numpy() + 1

n_entrenables = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
# =========================================================================
# 5 · Técnica A: cabeza lineal sobre [CLS] y sobre mean pooling
# =========================================================================
for i, resumen in enumerate(["[CLS]", "mean"]):
    lineal, seg = entrenar_cabeza(nn.Linear(768, 5), EMB["train"][i], y_tr_A, EMB["val"][i], y_va_A)
    nombre = f"A · BETO congelado + lineal ({resumen})"
    pred = predecir_cabeza(lineal, EMB["test"][i])
    # evaluar() heredada devuelve la fila registrada: le añadimos el costo propio de esta entrega
    evaluar(y_te_A, pred, nombre, n_params=N_PARAMS_BETO,
            segundos=seg + SEG_EXTRACCION)["n_entrenables"] = n_entrenables(lineal)
    PREDS[nombre] = pred

# Referencia barata: regresión logística sobre los mismos embeddings
# TODO(fase 3): LogisticRegression(class_weight="balanced") sobre EMB mean

<!-- LEER: ¿supera a TF-IDF (0,524)? ¿[CLS] o mean? Primera evidencia para H2. -->

---

## 6. Técnica B — BETO congelado + cabeza MLP propia

La cabeza del guía (768 → 512 → 256 → 5 con ReLU y dropout), con **una corrección**: el guía la
termina en `LogSoftmax`, pero `BertForSequenceClassification` aplica `CrossEntropyLoss`, que ya
incluye el `log_softmax`. Normalizar dos veces no rompe el entrenamiento pero aplana los
gradientes. Aquí la cabeza devuelve logits (`docs/DECISIONS.md` §D-304).

Pregunta: si el sentimiento está en el embedding pero no de forma lineal, una cabeza no lineal
debería extraerlo. Si apenas mejora, el sentimiento **no está** en la representación congelada.

In [ ]:
# =========================================================================
# 6 · Técnica B: cabeza MLP (la del guía, sin LogSoftmax)
# =========================================================================
def cabeza_mlp():
    return nn.Sequential(nn.Linear(768, 512), nn.ReLU(), nn.Dropout(0.2),
                         nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
                         nn.Linear(256, 5))

I_MEJOR = 1   # TODO(fase 3): elegir [CLS]=0 o mean=1 según validación de §5
mlp, seg = entrenar_cabeza(cabeza_mlp(), EMB["train"][I_MEJOR], y_tr_A, EMB["val"][I_MEJOR], y_va_A)
pred = predecir_cabeza(mlp, EMB["test"][I_MEJOR])
evaluar(y_te_A, pred, "B · BETO congelado + MLP", n_params=N_PARAMS_BETO,
        segundos=seg + SEG_EXTRACCION)["n_entrenables"] = n_entrenables(mlp)
PREDS["B · BETO congelado + MLP"] = pred

<!-- LEER: mejora de B sobre A. ¿Cabeza o representación? -->

---

## 7. Técnica C — *fine-tuning* completo

Ahora los 110 M de parámetros se ajustan a la tarea. Tres diferencias con el guía:

- **Pérdida ponderada por clase** (`WeightedTrainer`), igual que en todos los modelos anteriores.
- **Early stopping y selección del mejor modelo por macro-F1 de validación**, no el último.
- `lr = 2e-5` con warmup: aquí sí es la tasa correcta. En MP2 esa misma tasa era la trampa
  (§D-204), porque allí se partía de pesos aleatorios.

In [ ]:
# =========================================================================
# 7 · WeightedTrainer + utilidades de fine-tuning (reutilizadas en §9–§13)
# =========================================================================
import tempfile
from transformers import (AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          EarlyStoppingCallback)

class WeightedTrainer(Trainer):
    """Trainer con CrossEntropy ponderada por clase (el guía usa CE sin pesos)."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.functional.cross_entropy(out.logits, labels, weight=PESOS.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def compute_metrics(pred):
    y_pred = pred.predictions.argmax(-1) + 1; y_true = pred.label_ids + 1
    f = evaluar_silencioso(y_true, y_pred)
    return {"macro_f1": f["macro_f1"], "mae": f["mae"], "qwk": f["qwk"]}

def afinar(modelo, train, val, epocas, lr=CFG_BERT["lr"], tok=None, batch=CFG_BERT["batch"], optimizers=(None, None)):
    """Fine-tuning reproducible sin checkpoints en disco. Devuelve (trainer, segundos)."""
    fijar_semilla_hf()
    args = TrainingArguments(
        output_dir=tempfile.mkdtemp(), num_train_epochs=epocas, learning_rate=lr,
        per_device_train_batch_size=batch, per_device_eval_batch_size=CFG_BERT["batch_eval"],
        warmup_ratio=CFG_BERT["warmup_frac"], weight_decay=CFG_BERT["weight_decay"], fp16=CFG_BERT["fp16"],
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        logging_strategy="epoch", report_to="none", seed=SEED, data_seed=SEED)
    tr = WeightedTrainer(model=modelo, args=args, train_dataset=train, eval_dataset=val,
                         data_collator=DataCollatorWithPadding(tok or tokenizer),
                         compute_metrics=compute_metrics, optimizers=optimizers,
                         callbacks=[EarlyStoppingCallback(CFG_BERT["paciencia"])])
    t0 = time.time(); tr.train()
    return tr, time.time() - t0

def predecir_trainer(tr, dataset):
    return tr.predict(dataset).predictions.argmax(-1) + 1

In [ ]:
# =========================================================================
# 7 · Técnica C: fine-tuning completo de BETO
# =========================================================================
modelo_ft = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=5)
trainer_ft, seg_ft = afinar(modelo_ft, ds_tok["train"], ds_tok["val"], CFG_BERT["epocas"])

pred_ft = predecir_trainer(trainer_ft, ds_tok["test"])
# TODO(fase 3): latencia ms/reseña (lote 1 y lote 64)
evaluar(y_te_A, pred_ft, "C · BETO fine-tuning completo", n_params=n_entrenables(modelo_ft),
        segundos=seg_ft)["n_entrenables"] = n_entrenables(modelo_ft)
PREDS["C · BETO fine-tuning completo"] = pred_ft

In [ ]:
# --- Curvas de entrenamiento: pérdida train/val y macro-F1 de validación por época ---
# TODO(fase 3): a partir de trainer_ft.state.log_history

<!-- LEER: ¿supera a TF-IDF? ¿en qué época satura? ¿sobreajuste? -->

---

## 8. Comparación: tres técnicas, tres entregas

La tabla que responde H1 y H2. Las filas de MP1 y MP2 se toman de sus bitácoras
(`docs/EXPERIMENTS.md` de cada entrega): misma submuestra, mismo Split A, misma `evaluar`. Sus
tiempos provienen de otro hardware y se marcan como tales.

In [ ]:
# =========================================================================
# 8 · Tabla única + F1 por clase
# =========================================================================
PREVIOS = [  # de ../miniproyecto 1/docs/EXPERIMENTS.md y ../miniproyecto 2/docs/EXPERIMENTS.md
    dict(modelo="MP1 · TF-IDF + LogReg", macro_f1=0.5235, accuracy=0.6785, mae=0.376, qwk=0.7261,
         f1_1=0.596, f1_2=0.301, f1_3=0.433, f1_4=0.475, f1_5=0.813, n_entrenables=60_000, segundos=27.0),
    dict(modelo="MP1 · BiLSTM + atención", macro_f1=0.5274, accuracy=0.6758, mae=0.362, qwk=0.7546,
         n_entrenables=9_441_862, segundos=88.4),
    # TODO(fase 4): filas de MP2 (MLP y Transformer desde cero) cuando estén en su EXPERIMENTS.md
]
tabla = pd.DataFrame(PREVIOS + RESULTADOS)
cols = ["modelo", "macro_f1", "accuracy", "mae", "qwk"] + [f"f1_{e}" for e in ESTRELLAS] + ["n_entrenables", "segundos"]
tabla[cols].sort_values("macro_f1", ascending=False).round(4)

In [ ]:
# --- Matrices de confusión A / B / C lado a lado ---
# TODO(fase 4)

In [ ]:
# --- Costo-beneficio: parámetros entrenables (log) vs. macro-F1, todas las entregas ---
# TODO(fase 4)

<!-- LEER: veredicto H1 (¿gana en 2★/3★?) y H2 (¿congelado < TF-IDF?). ¿Compensa el costo? -->

---

> **Secciones 9 a 13 — estudios con configuración reducida.** Cada corrida usa `CFG_EST`: un
> subconjunto estratificado de 8.000 reseñas del train (mismo val y test) y menos épocas. Sus
> filas **se comparan entre sí**, no contra la tabla de la Sección 8.

In [ ]:
# --- Subconjunto de train para los estudios (estratificado, semilla fija) ---
def submuestra_train(n):
    if n >= len(idx_tr): return np.arange(len(idx_tr))
    sel, _ = train_test_split(np.arange(len(idx_tr)), train_size=n, random_state=SEED, stratify=y_tr_A)
    return np.sort(sel)

SEL_EST = submuestra_train(CFG_EST["n_train"])
ds_est = ds_tok["train"].select(SEL_EST)
ESTUDIOS = []   # filas de §9–§13
print(f"Train de estudios: {len(ds_est):,} reseñas")

## 9. ¿Cuántas capas hay que descongelar?

Entre congelar todo (§5) y afinarlo todo (§7) hay un continuo. Descongelamos solo las últimas
*k* capas del encoder (más la cabeza). Si con *k* = 2 se recupera casi todo el *fine-tuning*
completo, el sentimiento se aprende arriba y las capas bajas (sintaxis, morfología) sirven tal
cual.

Además probamos **LR discriminativa por capa** (*layer-wise learning rate decay*, Howard &
Ruder 2018, ULMFiT): cada capa hacia abajo recibe una tasa menor, para no destruir lo que menos
hay que cambiar. No se vio en clase.

In [ ]:
# =========================================================================
# 9 · Descongelar las últimas k capas
# =========================================================================
def congelar_hasta(modelo, k):
    """Deja entrenables solo las últimas k capas del encoder + pooler + clasificador."""
    base = modelo.base_model
    for p in base.parameters(): p.requires_grad = False
    capas = base.encoder.layer
    for capa in capas[len(capas) - k:] if k > 0 else []:
        for p in capa.parameters(): p.requires_grad = True
    if getattr(base, "pooler", None) is not None:
        for p in base.pooler.parameters(): p.requires_grad = True
    return modelo

for k in [0, 2, 4, 12]:
    m = congelar_hasta(AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=5), k)
    lr = CFG_BERT["lr_cabeza"] if k == 0 else CFG_BERT["lr"]
    tr, seg = afinar(m, ds_est, ds_tok["val"], CFG_EST["epocas"], lr=lr)
    f = evaluar(y_te_A, predecir_trainer(tr, ds_tok["test"]), f"§9 · k={k}", registrar=False)
    ESTUDIOS.append({**f, "estudio": "capas", "k": k, "n_entrenables": n_entrenables(m), "segundos": seg})
    del m, tr; torch.cuda.empty_cache()

In [ ]:
# --- LR discriminativa por capa (LLRD) ---
# TODO(fase 5): grupos de parámetros con lr * decay**(12 - i), decay≈0.9; pasar optimizers=(opt, None) a afinar()

In [ ]:
# --- Gráfica: macro-F1 y tiempo vs. k ---
# TODO(fase 5)

<!-- LEER: ¿dónde se satura? ¿aporta LLRD? -->

## 10. ¿Qué modelo preentrenado?

BETO es monolingüe. **mBERT** cubre 104 idiomas con el mismo tamaño, así que su vocabulario
dedica menos subpalabras al español y fragmenta más. **DistilBETO** tiene la mitad de capas.
Medimos cuánto vale ser monolingüe y cuánto cuesta comprimir.

In [ ]:
# =========================================================================
# 10 · BETO vs. mBERT vs. DistilBETO
# =========================================================================
CHECKPOINTS = {"BETO": CHECKPOINT,
               "mBERT": "google-bert/bert-base-multilingual-cased",
               "DistilBETO": "dccuchile/distilbert-base-spanish-uncased"}
for nombre, ckpt in CHECKPOINTS.items():
    tok = AutoTokenizer.from_pretrained(ckpt)
    fert, _ = fertilidad(tok, txt_tr[:2000])
    d = DatasetDict(train=ds["train"].select(SEL_EST), val=ds["val"], test=ds["test"]).map(
        tokenizar_ds(tok), batched=True, remove_columns=["text"])
    m = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=5)
    tr, seg = afinar(m, d["train"], d["val"], CFG_EST["epocas"], tok=tok)
    f = evaluar(y_te_A, predecir_trainer(tr, d["test"]), f"§10 · {nombre}", registrar=False)
    ESTUDIOS.append({**f, "estudio": "checkpoint", "checkpoint": nombre, "fertilidad": fert,
                     "n_entrenables": n_entrenables(m), "segundos": seg})
    del m, tr; torch.cuda.empty_cache()

<!-- LEER: ¿se traduce la fertilidad de mBERT en peor F1? ¿cuánto pierde DistilBETO por la mitad del costo? -->

## 11. Curva de eficiencia de datos

La promesa del preentrenamiento es necesitar menos datos etiquetados. Entrenamos BETO y TF-IDF
con 500, 2.000, 8.000 y 32.000 reseñas (mismo val/test) y marcamos el Transformer desde cero de
MP2 con sus 32.000. Responde H3.

In [ ]:
# =========================================================================
# 11 · Curva de eficiencia de datos: BETO FT vs. TF-IDF
# =========================================================================
# TODO(fase 5): para n in [500, 2000, 8000, 32000] (32000 = reutilizar §7 y MP1):
#   - TF-IDF + LogReg (misma configuración de MP1) sobre submuestra_train(n)
#   - BETO FT con CFG_EST["epocas"] (más épocas para n pequeños)
#   - gráfica log(n) vs. macro-F1, línea horizontal = Transformer MP2

<!-- LEER: veredicto H3. -->

## 12. LoRA: afinar sin tocar los pesos

*Low-Rank Adaptation* (Hu et al., 2021) congela BETO y aprende, en cada proyección de atención,
una corrección de rango bajo `W + BA`. Entrena < 1 % de los parámetros. No se vio en clase: es la
técnica con la que hoy se afinan los modelos grandes. ¿Se acerca al *fine-tuning* completo?

In [ ]:
# =========================================================================
# 12 · LoRA con peft
# =========================================================================
from peft import LoraConfig, get_peft_model, TaskType

cfg_lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
                      target_modules=["query", "value"])
m = get_peft_model(AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=5), cfg_lora)
m.print_trainable_parameters()
tr, seg = afinar(m, ds_est, ds_tok["val"], CFG_EST["epocas"], lr=5e-4)
f = evaluar(y_te_A, predecir_trainer(tr, ds_tok["test"]), "§12 · LoRA r=8", registrar=False)
ESTUDIOS.append({**f, "estudio": "lora", "n_entrenables": n_entrenables(m), "segundos": seg})
del m, tr; torch.cuda.empty_cache()

<!-- LEER: % entrenable vs. macro-F1 frente a k=0 y k=12 de §9. -->

## 13. Tarea de control: `Type`

Mismo procedimiento, prediciendo el tipo de establecimiento (hotel, restaurante, atracción): 3
clases balanceadas, no ordinal. Si BETO llega a ~0,95 como la BiLSTM de MP1, la dificultad de la
polaridad es de la tarea, no del modelo.

In [ ]:
# =========================================================================
# 13 · Tarea de control Type
# =========================================================================
# TODO(fase 6): etiquetas Type sobre SEL_EST/val/test, num_labels=3, sin pesos (balanceada),
#   macro-F1 propio (la evaluar() heredada está definida para 5 estrellas)

<!-- LEER: diferencia de macro-F1 Type vs. polaridad. -->

---

## 14. ¿Qué aprendió el *fine-tuning*?

Dos miradas. **Global:** proyectamos con UMAP el `[CLS]` de las reseñas de test antes (BETO
congelado) y después del *fine-tuning*, coloreado por estrellas. Si H2 es cierta, antes las
estrellas estarán mezcladas y después ordenadas. **Local:** *Integrated Gradients* (Sundararajan
et al., 2017) atribuye la predicción a cada token, sobre reseñas con negación.

Advertencia: una atribución indica a qué es sensible el modelo, no por qué decide.

In [ ]:
# =========================================================================
# 14.1 · UMAP del [CLS] antes y después del fine-tuning
# =========================================================================
# TODO(fase 6): EMB["test"][0] (antes) vs. [CLS] de trainer_ft.model.base_model (después); umap.UMAP(random_state=SEED)

In [ ]:
# =========================================================================
# 14.2 · Integrated Gradients sobre reseñas con negación
# =========================================================================
# TODO(fase 6): captum.attr.LayerIntegratedGradients sobre model.base_model.embeddings,
#   baseline = [PAD]; visualizar texto coloreado; contrastar con léxico del EDA §3.5

<!-- LEER: ¿se separan las estrellas tras el FT? ¿el modelo mira la negación? -->

---

## 15. Demo con pruebas de estrés

`predecir_resena(texto)` devuelve estrellas, distribución de probabilidad y atribución por
token. Ejecutamos **los mismos tres pares que en MP2 §15**, con texto fijo, para comparar
directamente con el Transformer desde cero. Evidencia anecdótica.

In [ ]:
# =========================================================================
# 15 · Demo
# =========================================================================
@torch.no_grad()
def predecir_resena(texto, modelo=None):
    modelo = (modelo or trainer_ft.model).eval()
    b = tokenizer(texto, truncation=True, max_length=MAX_LEN_BERT, return_tensors="pt").to(modelo.device)
    p = torch.softmax(modelo(**b).logits.float(), -1)[0].cpu().numpy()
    return int(p.argmax()) + 1, p

PARES = [("La comida estuvo buena.", "La comida no estuvo buena."),
         ("El hotel bien, la comida pésima.", "La comida pésima, el hotel bien."),
         ("Llegamos temprano, el lobby es amplio, la alberca limpia y el personal amable. "
          "La vista al mar es preciosa y el desayuno variado. Lástima que nos robaron en la habitación.",
          "Nos robaron en la habitación.")]
for a, b in PARES:
    for t in (a, b):
        e, p = predecir_resena(t)
        print(f"{e}★  {np.round(p, 2)}  «{t[:70]}»")
    print()
# TODO(fase 6): gráfica de barras de probabilidades + widget ipywidgets opcional

<!-- LEER: ¿responde a la negación? ¿al orden? ¿ve la queja final? Contraste con MP2. -->

---

## 16. Análisis de errores

Errores por distancia en estrellas, reseñas donde BETO falla y TF-IDF acierta (y al revés), y
categorías: ironía, reseña mixta, texto muy corto, etiqueta incoherente con el texto.

In [ ]:
# =========================================================================
# 16 · Análisis de errores
# =========================================================================
# TODO(fase 7): distribución |y - ŷ|; ejemplos por categoría; cruce con TF-IDF (reentrenar como en MP1)

<!-- LEER: tipos de error dominantes y si son del modelo o de la etiqueta. -->

---

## 17. Conclusiones y limitaciones

<!-- REDACTAR tras ejecutar:
- Respuesta a la pregunta: cuánto vale el preentrenamiento y qué hay que ajustar.
- Veredicto H1, H2, H3 con números.
- Recomendación práctica: qué técnica usaríamos en producción (F1 vs. costo vs. latencia).
- Limitaciones: estudios con CFG_EST, una sola semilla, tiempos de MP1/MP2 en otro hardware,
  etiquetas ruidosas, atribución no causal.
- Trabajo futuro: modelos de dominio (reseñas), pérdidas ordinales sobre BETO, aumento de datos
  de 1★/2★, destilación.
-->